In this notebook, you will train and evaluate your model on a. the test data you created from your time aware split b. a new test set that is generated after a new set of movies is released on the platform. 

Imports + define helper functions

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def dcg_at_k(rels, k=10):
    rels = np.asarray(rels)[:k]
    discounts = 1.0 / np.log2(np.arange(2, len(rels) + 2))
    return float(np.sum((2**rels - 1) * discounts))

def ndcg_for_group(df_group, score_col="score", label_col="watched", k=10):
    g = df_group.sort_values(score_col, ascending=False)
    dcg = dcg_at_k(g[label_col].values, k)
    ideal = dcg_at_k(np.sort(df_group[label_col].values)[::-1], k)
    return dcg / ideal if ideal > 0 else 0.0

def eval_ndcg(df, pipe, k=10):
    df = df.copy()
    X = df[["personalized_similarity","pred_completion_rate","recency_boost","genre"]]
    df["score"] = pipe.predict_proba(X)[:, 1]  # P(y=1)

    # Define ranking "queries" as (user_id, ts) slates
    overall = df.groupby(["user_id","ts"]).apply(ndcg_for_group, score_col="score", label_col="watched", k=k).mean()

    # Slice NDCG by genre: compute NDCG within slates but only for items of that genre
    # (This mimics “how well do we rank Indie items when they appear?”)
    by_genre = (
        df.groupby("genre")
          .apply(lambda g: g.groupby(["user_id","ts"]).apply(ndcg_for_group, score_col="score", label_col="watched", k=k).mean())
          .sort_values()
    )

    return float(overall), by_genre, df

Read train and test data

In [ ]:
ART = Path("../artifacts")

train = pd.read_parquet(ART / "train.parquet")
test1 = pd.read_parquet(ART / "test_data_pre_shift.parquet")
test2 = pd.read_parquet(ART / "test_data_post_shift.parquet")

for df in (train, test1, test2):
    df["ts"] = pd.to_datetime(df["ts"])

print("Rows:", {"train": len(train), "test1": len(test1), "test2": len(test2)})
print("Train  rate:", train["watched"].mean().round(4), "| Indie  rate:", train.loc[train["genre"]=="Indie","watched"].mean().round(4))
print("Test1  rate:", test1["watched"].mean().round(4), "| Test2  rate:", test2["watched"].mean().round(4))
print("Genre mix test1 (top):\n", test1["genre"].value_counts(normalize=True).head())
print("Genre mix test2 (top):\n", test2["genre"].value_counts(normalize=True).head())

# Features + preprocessing
num_cols = ["personalized_similarity","pred_completion_rate","recency_boost"]
cat_cols = ["genre"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

Train a model to predict "watched" from the features. 

In [ ]:

model = LogisticRegression(max_iter=200, n_jobs=None)

pipe = Pipeline(steps=[("prep", preprocess), ("clf", model)])

X_train = train[num_cols + cat_cols]
y_train = train["watched"].astype(int)

pipe.fit(X_train, y_train)

print("\nTrained LogisticRegression.")

Evaluate the model performance on both test sets. 

In [ ]:
ndcg1, ndcg1_by_genre, _ = eval_ndcg(test1, pipe, k=10)
ndcg2s, ndcg2s_by_genre, _ = eval_ndcg(test2, pipe, k=10)

slice_df = pd.concat(
    [ndcg1_by_genre.rename("ndcg_test1"), ndcg2s_by_genre.rename("ndcg_test2_shift")],
    axis=1
).reset_index().rename(columns={"index":"genre"})
slice_df["rel_change_%"] = 100*(slice_df["ndcg_test2_shift"] - slice_df["ndcg_test1"]) / slice_df["ndcg_test1"].replace(0, np.nan)

print("Overall NDCG@10 test1:", ndcg1)
print("Overall NDCG@10 test2_shift:", ndcg2s)
slice_df.sort_values("rel_change_%")

Why might the performance on Indie be higher? Plot the feature distributions below for each genre to determine the cause for the drop in performance. 

In [ ]:
import matplotlib.pyplot as plt

TARGET = "" #fill in with target genre

def boxplot_personalized_similarity(df, title):
    g = df[df["genre"] == TARGET]
    
    x_neg = g[g["watched"] == 0]["personalized_similarity"]
    x_pos = g[g["watched"] == 1]["personalized_similarity"]
    
    plt.figure()
    plt.boxplot([x_neg, x_pos], labels=["watched=0 (neg)", "watched=1 (pos)"])
    plt.title(title)
    plt.ylabel("personalized_similarity")
    plt.show()

boxplot_personalized_similarity(test1, f"Test1 — {TARGET} personalized_similarity by label")
boxplot_personalized_similarity(test2, f"Test2 Shifted — {TARGET} personalized_similarity by label")